# RAG with Citations, Abstention and Stage-wise Diagnosis

A RAG system that answers every question, with no sources, is hard to trust. This lab builds a **policy assistant** for the Refund Desk help center and adds three things that production RAG needs:

1. **Citations**: every answer names the passages it used, so a person (or a test) can check it.
2. **Abstention**: when the evidence is not in the corpus, the system says so instead of guessing. We treat this as a **decision model**: a yes/no *answerable?* probability plus a retrieval-score threshold.
3. **Stage-wise diagnosis**: when an answer is wrong, find out *which stage* failed (retrieval, ranking, generation or abstention) instead of tweaking prompts at random.

| Where it fits | Earlier notebook | What this lab adds |
| :--- | :--- | :--- |
| Hybrid retrieval and RRF | `6.7` (BM25 + dense on a paper corpus, then HotpotQA) | Same idea, in memory, on a small policy corpus; measured per question type |
| Re-ranking, HyDE, query transformations | `6.8`, `6.9` (query transformation, reranking) | Rerank score reused as an abstention signal |
| Guardrails and RAG-triad metrics | `6.11` (DeepEval) | Hand-built, per-stage metrics that tell you *where* to fix; no judge framework needed |
| Decision models (Week 8) | | Abstention is the first small decision model in the course |

The corpus (`policy_corpus.py`) has 24 policy passages and 22 labeled questions: direct, paraphrased, two-passage, and six that the corpus cannot answer (including a question about an *internal* refund limit that is deliberately not in any public text). The code lives in `code/projects/refund-desk/policy_rag.py` and is reused by the Refund Desk project. It uses its own small policy corpus on purpose: HotpotQA has no unanswerable questions, and abstention needs some. For the end-to-end pipeline on HotpotQA, see `6.13`.

**Prerequisites:** Ollama running locally with one chat model and one embedding model (`ollama pull llama3.2:3b && ollama pull nomic-embed-text`). No Qdrant or other server is needed. Optional: `sentence-transformers` for a cross-encoder reranker. To explore the code paths without Ollama, set `DESK_MOCK=1` before starting Jupyter (the mock models are rule-based, so quality numbers then describe the mock, not a real model).

In [ ]:
import os, sys
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

import rag_common as rc

cfg = rc.load_config()   # week 6 config.yaml
rc.check_services(cfg, qdrant=False)

# Pass config values to refund-desk llm module via environment variables
os.environ["OLLAMA_HOST"] = cfg["ollama"]["url"]
os.environ["OLLAMA_CHAT_MODEL"] = cfg["llm"]["model"]                               # chat model: same one the other RAG notebooks use
os.environ["OLLAMA_EMBED_MODEL"] = cfg["ollama"].get("embedding_model", "nomic-embed-text")   # an *Ollama* embedding model

PROJECT = Path("../projects/refund-desk").resolve()      # run from code/week6
sys.path.insert(0, str(PROJECT))
import llm, policy_rag as pr
from policy_corpus import PASSAGES, EVAL


## 1. Why citations and abstention

| Failure | What the user sees | Control |
| :--- | :--- | :--- |
| Retrieval misses the evidence | A fluent but wrong or generic answer | Better retrieval (hybrid, rerank); measure recall |
| Evidence is retrieved but buried | Answer uses a weaker passage | Ranking and context size |
| Model ignores or misreads the context | Confident claim the passage does not support | Citations plus a support check |
| Corpus has no answer | Invented policy | **Abstain** and hand off |

Citations make failures *visible*; abstention makes the last row *safe*; diagnosis tells you which row you are in.

In [ ]:
print(len(PASSAGES), "passages;", len(EVAL), "questions")
print(Counter(q["kind"] for q in EVAL))
for p in PASSAGES[:3]:
    print(p["id"], "|", p["title"], "|", p["text"][:90], "...")

## 2. Hybrid retrieval over the policy corpus

`PolicyIndex` builds a BM25 index (implemented from scratch, no dependency) and an embedding index (`llm.embed`), then fuses their rankings with **Reciprocal Rank Fusion** exactly as in `6.7`: $\text{RRF}(d)=\sum_m 1/(k+\text{rank}_m(d))$ with $k=60$. Because RRF uses ranks, BM25 scores and cosine similarities never need to be put on one scale.

We measure **recall@5** (fraction of the gold passages found in the top 5) and **MRR** (how high the first gold passage sits), only on the answerable questions, and break it down by question kind. Paraphrased questions share few words with their passage, so they are where the retrievers should disagree.

In [ ]:
index = pr.PolicyIndex()          # one passage per chunk; try max_words=25 later (exercise)
rows = []
for mode in ["bm25", "dense", "hybrid"]:
    rows.append(dict(retriever=mode, **pr.retrieval_metrics(index, mode=mode, k=5)))
pd.DataFrame(rows).round(3)

In [ ]:
def recall_by_kind(mode, rerank=None, k=5):
    out = {}
    for q in EVAL:
        if not q["gold"]: continue
        parents = [h["parent"] for h in index.search(q["question"], k=k, mode=mode, rerank=rerank)]
        out.setdefault(q["kind"], []).append(sum(g in parents for g in q["gold"]) / len(q["gold"]))
    return {kind: round(float(np.mean(v)), 2) for kind, v in out.items()}

pd.DataFrame({m: recall_by_kind(m) for m in ["bm25", "dense", "hybrid"]})

In [ ]:
# Where do the two retrievers disagree? Look at the paraphrase questions' gold-passage ranks.
rows = []
for q in EVAL:
    if q["kind"] != "paraphrase": continue
    hits = index.search(q["question"], k=len(index.chunks), mode="hybrid")
    gold = next(h for h in hits if h["parent"] == q["gold"][0])
    rows.append(dict(id=q["id"], question=q["question"][:55], gold=q["gold"][0], bm25_rank=gold["bm25_rank"], dense_rank=gold["dense_rank"],
                     hybrid_rank=hits.index(gold) + 1))
pd.DataFrame(rows)

## 3. Re-ranking, and the rerank score as a decision signal

A reranker re-scores the top candidates with a more careful model (a **cross-encoder** reads the query and passage together, as in `6.9`). We use it twice:

- to **re-order** the candidates (ranking quality), and
- as a **signal for abstention**: the best candidate's score tends to be lower when nothing in the corpus answers the question. That makes the score the cheapest decision input we have, because it costs no extra model call.

If `sentence-transformers` or the model download is unavailable, the notebook falls back to `overlap_reranker`, a simple query-term coverage score. Everything below runs either way.

In [ ]:
ce = pr.make_cross_encoder_reranker()
reranker = ce or pr.overlap_reranker
print("reranker:", "cross-encoder" if ce else "overlap fallback")

rows = [dict(retriever="hybrid", **pr.retrieval_metrics(index, "hybrid")),
        dict(retriever="hybrid + rerank", **pr.retrieval_metrics(index, "hybrid", rerank=reranker))]
pd.DataFrame(rows).round(3)

In [ ]:
# Does the top rerank score separate answerable from unanswerable questions?
sig = []
for q in EVAL:
    hits = index.search(q["question"], k=5, mode="hybrid", rerank=reranker)
    sig.append(dict(id=q["id"], kind=q["kind"], answerable=bool(q["gold"]), top_score=pr.retrieval_signal(hits)))
sig = pd.DataFrame(sig)
print(sig.groupby("answerable").top_score.describe().round(3))
print()
print(sig.sort_values("top_score").round(3).to_string(index=False))

Read the table: where the two groups overlap, a score threshold alone cannot separate them. That is why abstention also uses the model's own *answerable?* judgment (section 5). Also note the near-miss questions (price matching, internal refund limits): related passages exist, so the score can be deceptively high.

## 4. Answer with citations (schema-constrained)

Generation is constrained to a JSON schema with an `answer` and a list of `citations`, so a citation is data, not text we have to parse. We then keep only citations that point to passages that were actually in the context (a model can invent an id), and run a **support check**: a yes/no vote on whether the answer is supported by the cited text.

In [ ]:
for qid in ["Q05", "Q13", "Q17"]:
    q = next(x for x in EVAL if x["id"] == qid)
    r = pr.answer_with_citations(q["question"], index, rerank=reranker)
    print(qid, q["kind"], "| gold:", q["gold"])
    print("  Q:", q["question"])
    print("  A:", r["answer"])
    print("  cites:", r["citations"], "| abstained:", r["abstained"], "|", r["reason"])
    print("  scores:", {k: (round(v, 2) if v is not None else None) for k, v in r["scores"].items()}, "\n")

## 5. Abstention as a decision model

The system abstains when **either** signal says the corpus cannot answer:

1. **Retrieval signal** below `min_score` (free), or
2. **P(answerable)** below `p_min`: *k* yes/no votes from the chat model on "does this context answer the question?" (the same vote-fraction technique as notebook `8.9`).

To tune thresholds without paying for model calls again, we run every question **once with abstention switched off** (`p_min=0`), keep the raw signals, and then sweep thresholds offline. Positive class for the abstention decision = *unanswerable*. **Precision** = of the questions we refused, how many really had no answer; **recall** = of the unanswerable questions, how many we refused. The cost on the other side is the **false-abstain rate**: answerable questions we refused.

In [ ]:
raw = []
for q in EVAL:
    r = pr.answer_with_citations(q["question"], index, rerank=reranker, p_min=0.0, min_score=-1e9, votes=3)
    raw.append(dict(q=q, r=r, signal=r["scores"]["retrieval"], p=r["scores"]["p_answerable"]))
print("collected", len(raw), "raw results;", llm.CALLS["chat"], "chat calls so far")
pd.DataFrame([dict(id=x["q"]["id"], kind=x["q"]["kind"], signal=round(x["signal"], 3), p_answerable=x["p"]) for x in raw]).head(8)

In [ ]:
def abstain_decision(x, min_score, p_min):
    return x["signal"] < min_score or x["p"] < p_min

def sweep(min_scores, p_mins):
    rows = []
    for ms in min_scores:
        for pm in p_mins:
            pred = [abstain_decision(x, ms, pm) for x in raw]
            truth = [not x["q"]["gold"] for x in raw]
            tp = sum(p and t for p, t in zip(pred, truth)); fp = sum(p and not t for p, t in zip(pred, truth)); fn = sum((not p) and t for p, t in zip(pred, truth))
            prec = tp / (tp + fp) if tp + fp else 1.0; rec = tp / (tp + fn) if tp + fn else 1.0
            f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
            ans_total = sum(not t for t in truth)
            rows.append(dict(min_score=ms, p_min=pm, precision=round(prec, 2), recall=round(rec, 2), f1=round(f1, 2),
                             false_abstain_rate=round(fp / ans_total, 2) if ans_total else 0))
    return pd.DataFrame(rows)

scores = np.array([x["signal"] for x in raw])
grid = sweep(np.round(np.quantile(scores, [0.0, 0.25, 0.5, 0.75]), 3), [0.0, 0.34, 0.67, 1.0])
grid.sort_values(["f1", "false_abstain_rate"], ascending=[False, True]).head(10)

In [ ]:
best = grid.sort_values(["f1", "false_abstain_rate"], ascending=[False, True]).iloc[0]
MIN_SCORE, P_MIN = float(best.min_score), float(best.p_min)
print(f"operating point: min_score={MIN_SCORE}, p_min={P_MIN}  ->  precision={best.precision}, recall={best.recall}, "
      f"false-abstain rate={best.false_abstain_rate}")

**Choosing the operating point is a product decision, not just an F1 maximization.** A false answer (inventing a policy) and a false abstention (sending a customer to a person for a question the policy answers) cost different amounts. A support desk that hands off cheaply may prefer high recall on *unanswerable*; a self-serve help page may prefer a low false-abstain rate. Change the objective in the cell above and see how the operating point moves.

## 6. Stage-wise diagnosis

For each question we record what happened at every stage and label the **first** stage that went wrong:

| Stage | Check | Failure label |
| :--- | :--- | :--- |
| Retrieval | all gold passages in the top 5 | `retrieval error` |
| Ranking / context | all gold passages in the top 3 that the generator saw | `ranking error` |
| Abstention | abstained iff the question is unanswerable | `abstention error` (false abstain or false answer) |
| Generation | citations include the gold passages and nothing else; support check passes | `generation error` |

The labels tell you what to change: retrieval errors call for better chunking or retrieval; ranking errors for reranking or a larger context; generation errors for prompt or model changes; abstention errors for thresholds.

In [ ]:
def diagnose(x, min_score=MIN_SCORE, p_min=P_MIN):
    q, r = x["q"], x["r"]
    gold = set(q["gold"])
    hits = {h.split("#")[0] for h in r["hits"]}; ctx = {h.split("#")[0] for h in r["context"]}
    abstained = abstain_decision(x, min_score, p_min)
    if not gold:
        return "ok (correctly abstained)" if abstained else "abstention error: answered an unanswerable question"
    if not gold <= hits: where = "retrieval error: gold passage not retrieved"
    elif not gold <= ctx: where = "ranking error: gold below the context cutoff"
    else: where = None
    if abstained:
        return where or "abstention error: abstained although the context had the evidence"
    if where: return where
    cites = {c.split("#")[0] for c in r["citations"]}
    if not cites & gold: return "generation error: missing or wrong citation"
    if not gold <= cites: return "generation error: incomplete citations"
    if not cites <= gold: return "ok (extra citation)"
    if r["scores"]["p_supported"] is not None and r["scores"]["p_supported"] < 0.5: return "generation error: unsupported claim"
    return "ok"

diag = pd.DataFrame([dict(id=x["q"]["id"], kind=x["q"]["kind"], question=x["q"]["question"][:50], outcome=diagnose(x)) for x in raw])
diag["stage"] = diag.outcome.str.split(":").str[0].str.replace(r" \(.*\)", "", regex=True)
diag.sort_values("stage").to_string(index=False)

In [ ]:
print(pd.crosstab(diag.kind, diag.stage))
print()
print(diag.stage.value_counts())

In [ ]:
# Failing questions with their signals, for debugging
bad = diag[~diag.outcome.str.startswith("ok")]
for _, row in bad.iterrows():
    x = next(v for v in raw if v["q"]["id"] == row.id)
    print(row.id, row.outcome)
    print("   Q:", x["q"]["question"], "| gold:", x["q"]["gold"])
    print("   hits:", x["r"]["hits"], "| context:", x["r"]["context"], "| cites:", x["r"]["citations"],
          "| signal=%.2f p_answerable=%.2f" % (x["signal"], x["p"]))

## 7. Exercises and next steps

1. **Chunking.** Rebuild with `pr.PolicyIndex(max_words=25, overlap=8)` and rerun sections 2 and 6. Does recall change? Do citations still point to a parent passage (`P01#1` becomes `P01`)? What happens to the support check when a passage is split in two?
2. **Rerank.** If you used the overlap fallback, install `sentence-transformers` and rerun with the real cross-encoder. Which diagnosis labels shrink?
3. **Thresholds.** Re-pick the operating point for a use case that cannot tolerate invented policy (maximize recall on unanswerable subject to false-abstain rate under 20 percent).
4. **Query rewriting.** Add HyDE or multi-query from `6.8` in front of `index.search`. Do paraphrase questions improve?
5. **Refund Desk.** The project's `search_docs` MCP tool is a keyword match. Replace it with `answer_with_citations` and make the agent hand off to a human when the tool abstains. That keeps *abstain* in the same vocabulary as the Week 8 guards.

## Summary & Key Takeaways

1. **Citations are data**: a schema-constrained `citations` list lets you test grounding automatically.
2. **Abstention is a decision**: combine a free retrieval signal with a yes/no probability, tune on stored raw signals, and choose the operating point by the cost of each error.
3. **Diagnose by stage**: recall@k, context inclusion, citation correctness and abstention accuracy point at different fixes.
4. **Near-misses are the hard case**: related passages make scores look good while the answer is absent.
5. Guardrails and RAG-triad metrics (`6.11`) tell you *that* quality dropped; stage-wise diagnosis tells you *where*.

Next: `6.13-RAG-Capstone-End-to-End.ipynb` runs the whole pipeline on HotpotQA; then do `Exercise-6.1.md` on raw PDFs.
